# LTLTRA001 | Tracey Letlape | CSC3042F

## Multi-Layer Perceptron (MLP) vs Convolutional Neural Network (CNN)

Purpose of the assignment is to investigate the MLP and CNN architectures throught the following complementary lenses:
* Complexity Ladder: When MLP begin to degrade and why CNNs maintain performane
* Parameter Budget: Fix computational budget, and determine how much architecture design matters for each model class.

In [ ]:
# Necessary imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split, Subset
import numpy as np
import random

# DATA LOADING

* Define a transformer for each dataset
* Get both the training and the test datasets
* Split the training dataset into the training and validation datasets

In [ ]:
# Define a reusable helper method to split a dataset
def _split_dataset(dataset: datasets.MNIST | datasets.CIFAR10, train_ratio: float = 0.8, seed: int = 42) -> tuple[Subset, Subset]:
    # Compute split sizes
    total_size = len(dataset)

    train_size = int(train_ratio * total_size)
    val_size = total_size - train_size

    # Reproducible split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size]
    )

    return train_dataset, val_dataset

In [ ]:
# Define MNIST normalisation parameters
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.30811,)

# Define a MNIST transfom
MNIST_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
    transforms.Lambda(lambda x: x.view(-1))
])

# Get the MNIST datasets
mnist_full_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=MNIST_transform
)

mnist_test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=False,
    transform=MNIST_transform
)

# Split the training dataset into train and val datasets
mnist_train_dataset, mnist_val_dataset = _split_dataset(mnist_full_train_dataset)

print(f"Train size: {len(mnist_train_dataset)}")
print(f"validation size: {len(mnist_val_dataset)}")
print(f"Test size: {len(mnist_test_dataset)}")

In [ ]:
# Define CIFAR Normalisation parameters
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Define a CIFAR train transform
CIFAR_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Define a CIFAR test transform
CIFAR_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

cifar_full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=CIFAR_train_transform
)

cifar_test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=CIFAR_test_transform,
)

# Split the training dataset into train and val datasets
cifar_train_dataset, cifar_val_dataset = _split_dataset(cifar_full_train_dataset)

print(f"Train size: {len(cifar_train_dataset)}")
print(f"validation size: {len(cifar_val_dataset)}")
print(f"Test size: {len(cifar_test_dataset)}")

# PART A

* Implementation: Code for both architectures and training loop
* Results table: Test Accuracy vs Dataset combinations
* Training Curves: Loss and accuracy plots on a single figure
* Analysis


In [ ]:
# Reproducibility
seed = random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

# GLOBAL VARIABLES
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 10
MNIST_CLASSES = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9') 
CIFAR_CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## Data Loading - MNIST

## Data Loading - CIFAR-10

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

CIFAR_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

CIFAR_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

cifar_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=CIFAR_train_transform
)

cifar_test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=CIFAR_test_transform,
)

## Multi-Layer Perceptron
* Input Layer, 3 Hidden Layers, Output Layer
* Architecture: [input_dim -> 256 -> 128 -> 64 -> 32 -> 10]
* Uses the ReLU activation function for all layers except the output layer which uses sigmoid.

In [ ]:
# Define a fixed MLP
class MLP(nn.Module):
    """
        Feedforward network implementing MLP.
        Architecture: [input_dim -> *hidden_dims -> output_dim]
    """
    def __init__(self, input_dim, num_classes: int = 10):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, 256)
        self.h1 = nn.Linear(256, 128)
        self.h2 = nn.Linear(128, 64)
        self.h3 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):
        # Input layer
        self.z1 = self.input_layer(x)
        self.a1 = F.relu(self.z1)

        # First hidden layer
        self.z2 = self.h1(self.a1)
        self.a2 = F.relu(self.z2)

        # Second hidden layer
        self.z3 = self.h1(self.a2)
        self.a3 = F.relu(self.z3)

        # Third hidden layer
        self.z4 = self.h1(self.a3)
        self.a4 = F.relu(self.z4)

        # Output layer
        self.z5 = self.h1(self.a4)
        self.a5 = F.sigmoid(self.z4)

        return self.a5

## Convolutional Neural Network (CNNs)
* Input Layer, 3 Hidden Layers, Output Layer
* Architecture: [input_dim -> 256 -> 128 -> 64 -> 32 -> 10]
* Uses the ReLU activation function for all layers except the output layer which uses sigmoid.

In [ ]:
class ConvBlock(nn.Module):
    """
    Conv2d -> Batchnorm -> ReLU
    """

    def __init__(self, in_ch, out_ch, kernel, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel,
                      stride=stride, padding=padding, bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_channels, dropout=0.5, num_classes=10):
        super().__init__()

        # ── Stage 1: extract low-level features ───────────────────────────
        self.stage1 = nn.Sequential(
            ConvBlock(input_channels,  32),      # Transformation into desired shape
            ConvBlock(32, 32),      # Reform
            nn.MaxPool2d(2, 2),             # HxW -> H/2 x W/2
        )

        # ── Stage 2: higher-level features ────────────────────────────────
        self.stage2 = nn.Sequential(
            ConvBlock(32, 64),
            ConvBlock(64, 64),
            nn.MaxPool2d(2, 2),             # H/2 x W/2 -> H/4 x W/4
        )

        # ── Stage 3: abstract representations ─────────────────────────────
        self.stage3 = nn.Sequential(
            ConvBlock(64, 128),
        )

        # ── Classifier head ───────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),       # 128xH'xW' -> 128x1x1
            nn.Flatten(),                       #  -> 128
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.head(x)
        return x

## Training Loop

In [ ]:
def train_model(model: MLP | CNN, loader: DataLoader, criterion: nn.Module, optimiser: optim.Optimizer, device: torch.device | str = 'cpu'):
    # Move model to device
    model.to(device)
    
    # Set model to training mode
    model.train()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Reset gradients to zero
        optimiser.zero_grad()

        # Forward pass & loss computation
        logits = model(images)
        loss = criterion(logits, labels)

        # Backward pass & parameter update
        loss.backward()
        optimiser.step()

        # Update metrics
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

    # Compute avarage metrics
    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc

@torch.no_grad()
def evaluate_model(model: MLP | CNN, loader: DataLoader, criterion: nn.Module, device: torch.device | str = 'cpu'):
    # Move model to device
    model.to(device)
    
    # Set the model to evaluation mode
    model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        logits = model(images)
        loss = criterion(logits, labels)

        # Update metrics
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

    # Compute avarage metrics
    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc


def training_loop(model_name: str, train_dataset: datasets.MNIST | datasets.CIFAR10, test_dataset: datasets.MNIST | datasets.CIFAR10, lr: float = 1e-3, batch_size: int = 128, dropout: float = 0.5, num_epochs: int = 10, weight_decay: float = 1e-4, device: torch.device | str = 'cpu'):
    # Use pin_memory only if using CUDA
    pin_memory = device == 'cuda' or str(device).startswith("cuda")
    
    # Define loader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin_memory)
    
    # Get input_size
    temp, _ = next(iter(train_loader))

    # Define the model
    model = None
    if model_name == 'MLP':
        input_size = temp[0].numel()
        model = MLP(input_size, NUM_CLASSES)

    elif model_name == 'CNN':
        input_channels = temp.shape[1]
        model= CNN(input_channels=input_channels, dropout=dropout, num_classes=NUM_CLASSES)

    else:
        raise ValueError(f"model_name must be either 'MLP' or 'CNN'")
    
    # Move model to device
    model = model.to(device)
    
    # Use CrossEntropyLoss as the loss function
    criterion = nn.CrossEntropyLoss()

    # Use AdamW with weight_decay as 1e-4 as the optimiser
    optimiser = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Define a scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)

    # Store metrics
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }

    # Run through epochs
    for epoch in range(1, num_epochs+1):
        tr_loss, tr_acc = train_model(model, train_loader, criterion, optimiser, DEVICE)
        va_loss, va_acc = evaluate_model(model, test_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        print(f'Epoch {epoch:02d}/{num_epochs}  '
          f'Train loss={tr_loss:.4f} acc={tr_acc:.3f}  '
          f'Val loss={va_loss:.4f} acc={va_acc:.3f}  '
          f'lr={scheduler.get_last_lr()[0]:.6f}')
        
    return model, history